# **Deeploc 2.1**

2025 겨울 URP / 문서연, 김대현, 권효재


---


여기서부턴 Deeploc 2.1과 같은 구조로 학습을 진행!


---


개선/공부 필요(2026_01_18):

---

# **라이브러리 설명**

# torch
Pytorch 라이프러리 패키지


* torch.autograd: 자동 미분을 위한 함수가 포함됨(ex. enable/no_grad: 자동 미분 on/off, Function: 자체 미분 함수 정의 클래스)
* torch.nn: 신경망 구축을 위한 기본 데이터 구조/레이어(RNN/LSTM)/활성화 함수(ReLU)/손실 함수(MSELoss) 포함됨
* torch.optim: 확률적 경사 하강법(Stochastic Gradient Descent, SGD) 중심의 파라미터 옵티마이저 알고리즘
* torch.utils.data: SDG 반복연산 시에 사용하는 미니배치 유틸리티 포함됨
* torch.onnx: ONNX(Open Neural Network Exchange) 포맷으로 모델 export 할 때 사용

In [7]:
# 필요 라이브러리 설치 및 import
#!pip install -q torch pandas numpy safetensors optuna scikit-learn #lmdb

import io
import gc
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import math

from tqdm import tqdm
from safetensors import safe_open
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score, accuracy_score, matthews_corrcoef

In [ ]:
# <저장소 설정>

# Colab 환경
# from google.colab import drive

# drive.mount("/content/drive")
# SAVE_PATH = "/content/drive/MyDrive/Github/Mprotein_hydrophobic/dataset/lmdb"

# 로컬 환경
SAVE_PATH = "./"

os.makedirs(SAVE_PATH, exist_ok=True)

In [6]:
# 데이터셋 클래스 정의
class K_CV_Dataset(Dataset):
    # 데이터셋 전처리
    def __init__(self, K_CV, validation_k, save_path, is_train=True):
        # 기본 저장 경로 & 핸들 초기화
        self.save_path = save_path
        self.path_x = os.path.join(self.save_path, f"embeddings.safetensors")
        self.path_y = os.path.join(self.save_path, f"targets.safetensors")
        self.handle_x = None

        # 훈련/테스트 > 훈련 파티션 숫자 리스트 만들기
        if is_train:
            self.train_part = [i for i in range(K_CV) if i != validation_k]
        else:
            self.train_part = [validation_k]

        # 키 필터링 + 재현성을 위해 정렬
        self.train_keys = []
        self.targets_dict = {}
        with safe_open(self.path_y, framework="pt") as f:
            all_keys = f.keys()
            for p in self.train_part:
                prefix = f"part_{p}_"
                p_keys = [k for k in all_keys if k.startswith(prefix)]
                self.train_keys.extend(p_keys)
                for k in p_keys:
                    self.targets_dict[k] = f.get_tensor(k)
        self.train_keys.sort()

    # 몇개있는지 알려줘야함
    def __len__(self):
        return len(self.train_keys)

    def open_files(self):
        # 멀티프로세싱의 각 worker 안에서 파일을 처음 한 번만 엽니다. > 뭔말인지 잘 이해 못함 솔직히 num worker 각각이 핸들 한번씩 연다는말같긴한데..
        if self.handle_x is None:
            self.handle_x = safe_open(self.path_x, framework="pt")

    # 데이터셋 샘플 1개 가져오기
    def __getitem__(self, idx):
        self.open_files()

        # 리스트에서 idx번째 key 호출
        key = self.train_keys[idx]

        # get_slice().asarray() 또는 get_tensor() 사용
        # Safetensors는 메모리 매핑을 쓰기 때문에 이 과정이 매우 빠릅니다.
        embedding = self.handle_x.get_tensor(key)
        target = self.targets_dict[key]

        return embedding, target


# 작동 테스트
# c = K_CV_Dataset(4, 0, SAVE_PATH)
# a, b = c[0]
# print(a)
# print(b)

In [ ]:
import time

c = K_CV_Dataset(4, 0, SAVE_PATH)

# 테스트1: 순차 접근
st = time.time()
for i in range(100):
    c[i]
print(f"순차 100개: {time.time()-st:.2f}s")

# 테스트2: 셔플 접근
import random

random.shuffle(indices := list(range(len(c))))
st = time.time()
for i in indices[:100]:
    c[i]
print(f"셔플 100개: {time.time()-st:.2f}s")

In [ ]:
def padding_collate_fn(batch):
    # 임베딩/타겟 분리
    embeddings = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    # 2. 임베딩 패딩
    padded_embeddings = pad_sequence(embeddings, batch_first=True, padding_value=0)
    targets = torch.stack(targets)
    return padded_embeddings, targets

In [ ]:
# 앞에서 Layer normalization 해준걸 인풋으로 받는다고 가정
class MultiheadAttentionPooling(nn.Module):
    def __init__(self, attn_dim=128, heads_nums=2, kernel_size=5):
        super().__init__()
        self.attn_dim = attn_dim
        # 우선은,, 멀티헤드로 구현을 한다
        self.heads_nums = heads_nums
        # 멀티헤드의 디멘션은 전체 디멘션을 헤드 개수로 나눈것
        # 이거때문에 헤드 개수를 잘 나눠지도록(?) 설정함 보통
        self.head_dim = attn_dim // heads_nums

        # Q, K = V 만들기
        # Q : learnable Query, 먼저 (1, 1, attn_dim) 에 해당하는 빈 벡터 > xavier_uniform_ 하면 입출력 고려해서 난수생성 가능
        self.query = nn.Parameter(torch.empty(1, 1, attn_dim))
        nn.init.xavier_uniform_(self.query.data)
        self.w_kv = nn.Linear(
            attn_dim, attn_dim
        )  # 입력/출력 크기가 attn_dim인 Linear FC를 수행하는 모듈 // key value 짜피 같으니까 이걸로 한번에 할거고 논문도 그렇게 했는데 둘이 따로 초기화한다면? 즉 파라미터가 두개라면..?

    def forward(self, x):
        # 입력으로 받을 형태: (batch_size, sequence_length, attn_dim)
        batch_size, sequence_length, _ = x.shape

        # 배치 사이즈에 맞게 복제: 각 배치 샘플 전부 같은 쿼리 파라미터 공유
        # attn_dim을 헤드별로 쪼갬()
        # 헤드별로 계산할거라서 헤드를 앞으로 뺌
        Q = (
            self.query.repeat(batch_size, 1, 1)
            .view(batch_size, 1, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        K = (
            self.w_kv(x)
            .view(batch_size, sequence_length, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        V = (
            self.w_kv(x)
            .view(batch_size, sequence_length, self.heads_nums, self.head_dim)
            .transpose(1, 2)
        )
        # 셋다 (batch_size, head_nums, sequence_length(Q:1; per token), head_dim)
        # print(Q)
        # print(K)
        # print(V)
        # 행렬곱 > Attention Scalar score 구함!
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # print(f"어텐션 Score: {scores.shape}")
        # Conv1d 가우시안 필터
        scores_c1d_gaussian = scores

        # Softmax 적용
        attn_weights = F.softmax(scores_c1d_gaussian, dim=-1)
        # print(f"어텐션 Weights: {attn_weights.shape}")

        # 행렬곱 > weighted attention
        attnpooled_nosum = torch.matmul(attn_weights, V)
        # print(f"어텐션 Pooling (합하기 전): {attnpooled_nosum.shape}")

        # 가중합을 위해 (batch_size, head_nums, 1, head_dim) 순서니까
        # 1 없애고 reshape로 head concat해주기
        attentionpooled = attnpooled_nosum.squeeze(2).reshape(batch_size, self.attn_dim)
        # print(f"어텐션 Pooling: {attentionpooled.shape}")

        return attentionpooled

In [ ]:
# import matplotlib.pyplot as plt

# sigma1 = 3
# sigma2 = 50


# def gaussian_filter1d(size, sigma):
#     filter_range = np.linspace(-int(size / 2), int(size / 2), size)
#     gaussian_filter = [
#         1 / (sigma * np.sqrt(2 * np.pi)) * np.exp(-(x**2) / (2 * sigma**2))
#         for x in filter_range
#     ]
#     return gaussian_filter


# fig, ax = plt.subplots(1, 2)
# ax[0].plot(gaussian_filter1d(size=365, sigma=sigma1))
# ax[0].set_title(f"sigma= {sigma1}")
# ax[1].plot(gaussian_filter1d(size=365, sigma=sigma2))
# ax[1].set_title(f"sigma= {sigma2}")
# plt.show()

In [ ]:
class Deeploc2_1(nn.Module):
    def __init__(self, heads_nums, embedding_dim, attn_dim, output_dim=4, hidden_dim=0):
        super().__init__()
        self.input_layer_normalization = nn.LayerNorm(embedding_dim)
        self.input_linear_fc = nn.Linear(embedding_dim, attn_dim)
        self.attention_layer_normalization = nn.LayerNorm(attn_dim)
        self.attention_head = MultiheadAttentionPooling(
            attn_dim=attn_dim, heads_nums=heads_nums
        )
        self.dropout = nn.Dropout(0.1)
        if hidden_dim == 0:
            hidden_dim = attn_dim
        self.output_linear_fc = nn.Linear(
            hidden_dim, output_dim
        )  # mlp층 쌓으면 attn_dim 아니고 hidden_dim 됨 이거 나중에 고쳐야할듯

    def forward(self, x):
        # layer normalization, 선형층 통과 > embedding dim에서 attn dim으로,,
        x = self.input_layer_normalization(x)
        x = self.input_linear_fc(x)
        # attention head 통과 > (batch_size, attn_dim)
        x = self.attention_layer_normalization(x)
        x = self.attention_head(x)
        x = self.dropout(x)
        # 선형층 통과(분류기) > attn dim에서 4개로,,
        x = self.output_linear_fc(x)
        return x

NameError: name 'nn' is not defined

In [ ]:
# @title
def sigmoid_focal_loss(
    inputs: torch.Tensor,
    targets: torch.Tensor,
    alpha: float = 0.25,
    gamma: float = 2,
    reduction: str = "none",
) -> torch.Tensor:

    if not (0 <= alpha <= 1) and alpha != -1:
        raise ValueError(
            f"Invalid alpha value: {alpha}. alpha must be in the range [0,1] or -1 for ignore."
        )

    p = torch.sigmoid(inputs)
    ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
    p_t = p * targets + (1 - p) * (1 - targets)
    loss = ce_loss * ((1 - p_t) ** gamma)

    if alpha >= 0:
        alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
        loss = alpha_t * loss

    # Check reduction option and return loss accordingly
    if reduction == "none":
        pass
    elif reduction == "mean":
        loss = loss.mean()
    elif reduction == "sum":
        loss = loss.sum()
    else:
        raise ValueError(
            f"Invalid Value for arg 'reduction': '{reduction} \n Supported reduction modes: 'none', 'mean', 'sum'"
        )
    return loss

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
K_CV = 4
NUM_EPOCHS = 5

In [ ]:
DATASET_TRAIN = K_CV_Dataset(K_CV, 3, SAVE_PATH)
DATASET_TEST = K_CV_Dataset(K_CV, 3, SAVE_PATH, is_train=False)
DATALOADER_TRAIN = DataLoader(
    DATASET_TRAIN,
    batch_size=5,
    shuffle=True,
    num_workers=2,
    collate_fn=padding_collate_fn,
    pin_memory=True,
    persistent_workers=True,
)
DATALOADER_TEST = DataLoader(
    DATASET_TEST,
    batch_size=5,
    shuffle=False,
    num_workers=2,
    collate_fn=padding_collate_fn,
    pin_memory=True,
    persistent_workers=True,
)
MODEL = Deeploc2_1(embedding_dim=1152, attn_dim=128, output_dim=4, heads_nums=2).to(
    DEVICE
)
OPTIMIZER = AdamW(MODEL.parameters(), lr=1e-4)

for epoch in range(NUM_EPOCHS):
    # TRAIN
    MODEL.train()
    for embeddings, targets in tqdm(DATALOADER_TRAIN):
        OPTIMIZER.zero_grad()
        embeddings, targets = embeddings.to(DEVICE), targets.to(DEVICE)
        outputs = MODEL(embeddings)
        loss_value = sigmoid_focal_loss(
            outputs, targets, alpha=0.25, gamma=2, reduction="mean"
        )
        loss_value.backward()
        OPTIMIZER.step()

    # TEST
    MODEL.eval()
    val_loss = 0
    all_probs = []
    all_targets = []
    with torch.no_grad():
        # 여기서 주의: 테스트/검증용 데이터로더(VAL_DATALOADER)가 따로 있어야 합니다!
        # 만약 지금 DATALOADER 하나로만 하신다면, 아래는 '학습 데이터에 대한 점수'가 됩니다.
        for embeddings, targets in DATALOADER_TEST:
            embeddings, targets = embeddings.to(DEVICE), targets.to(DEVICE)
            outputs = MODEL(embeddings)

            # Loss 계산
            v_loss = sigmoid_focal_loss(
                outputs, targets, alpha=0.25, gamma=2, reduction="mean"
            )
            val_loss += v_loss.item()

            # 확률값 변환 (sigmoid) 및 저장
            probs = torch.sigmoid(outputs)
            all_probs.append(probs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # 리스트를 하나로 합치기 (Batch 단위 -> 전체 데이터 단위)
    all_probs = np.concatenate(all_probs, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    from sklearn.metrics import accuracy_score, f1_score

    # 1. 일단 0.5 기준으로 예측값(0 또는 1) 확정
    all_preds = (all_probs >= 0.5).astype(int)

    # 2. Accuracy 계산
    # Multi-label에서 accuracy_score는 모든 라벨이 정답과 정확히 일치해야 1로 계산됩니다 (엄격함)
    acc = accuracy_score(all_targets, all_preds)

    # 3. F1-score 계산
    # macro: 클래스별 F1을 구한 뒤 산술 평균 (클래스별 불균형이 클 때 추천)
    # micro: 전체 샘플의 TP, FN, FP를 다 합쳐서 계산
    f1_macro = f1_score(all_targets, all_preds, average="macro")
    f1_micro = f1_score(all_targets, all_preds, average="micro")

    print(
        f"Accuracy: {acc:.4f} | F1 (Macro): {f1_macro:.4f} | F1 (Micro): {f1_micro:.4f}"
    )

    # 0.5 기준으로 간단하게 MCC 계산 (클래스별 평균)
    mcc_scores = []
    for i in range(all_targets.shape[1]):  # output_dim=4니까 4번 돔
        mcc = matthews_corrcoef(all_targets[:, i], (all_probs[:, i] >= 0.5).astype(int))
        mcc_scores.append(mcc)

    avg_mcc = np.mean(mcc_scores)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Val Loss: {val_loss/len(DATALOADER_TEST):.4f}"
        f"Avg MCC: {avg_mcc:.4f} | "
        f"F1-Macro: {f1_macro:.4f} | "
        f"Acc: {acc:.4f}"
    )
del MODEL, DATALOADER_TRAIN, DATALOADER_TEST
torch.cuda.empty_cache()

  7%|▋         | 237/3450 [00:05<01:12, 44.13it/s]


KeyboardInterrupt: 

In [ ]:
# 데이터셋 클래스 정의
class K_CV_MultipleFiles_Dataset(Dataset):
    # 데이터셋 전처리
    def __init__(self, K_CV, validation_k, save_path, is_train=True):
        # 훈련 > K개의 데이터셋 중 validation_k 제외한 리스트 생성
        # 테스트 > validation_k만 생성
        if is_train:
            self.train_part = [i for i in range(K_CV) if i != validation_k]
        else:
            self.train_part = [validation_k]

        # 기본 저장 경로: save_path
        self.save_path = save_path

        # x와 y는 같은 key(ACC) 공유
        # y로부터 총합 key list를 만들 예정
        # key가 x의 어떤 파티션에 있는지 불러오기 위해 key와 x 파일 주소가 저장된 일종의 주소록 딕셔너리 생성
        # 반복문을 돌며 총합 targets 딕셔너리 생성
        self.key_list = []
        self.targets = {}
        self.keys_to_x = {}
        self.file_handels = {}

        for i in self.train_part:
            # x, y 경로 설정
            path_x = os.path.join(self.save_path, f"embeddings_part_{i}.safetensors")
            path_y = os.path.join(self.save_path, f"target_part_{i}.pt")

            ##타겟 파일
            # 타겟 파일 로드, 딕셔너리 업데이트, keys 리스트 가져오기
            y = torch.load(path_y, weights_only=False)
            self.targets.update(y)
            key_list_k = list(y.keys())
            self.key_list.extend(key_list_k)

            ##딕셔너리 주소 저장
            for key in key_list_k:
                self.keys_to_x[key] = path_x

        # 재현성을 위해 정렬
        self.key_list.sort()

    # 몇개있는지 알려줘야함
    def __len__(self):
        return len(self.key_list)

    # 데이터셋 샘플 1개 가져오기
    def __getitem__(self, idx):
        # 리스트에서 idx번째 key 호출
        key = self.key_list[idx]

        # 1. 이 키가 위치한 파일이 어디인지 주소보고 확인
        path_x_k = self.keys_to_x[key]

        # 2. 파일 핸들 열려있는지 확인, 안 열려있으면 열어서 보관함에 등록
        if path_x_k not in self.file_handels:
            self.file_handels[path_x_k] = safe_open(
                path_x_k, framework="pt", device="cpu"
            )

        # 3. 보관함에서 핸들 꺼내오기
        f = self.file_handels[path_x_k]

        embedding = f.get_tensor(key).clone()
        embedding = embedding.squeeze(0)

        target_numpy = self.targets[key]
        target = torch.from_numpy(target_numpy).squeeze(0).float()

        return embedding, target


# 작동 테스트
# c = K_CV_MultipleFiles_Dataset(4, 0, SAVE_PATH)
# a, b = c[0]
# print(a)
# print(b)

In [ ]:
# 데이터셋 클래스 정의
# 훈련 > K개의 데이터셋 중 validation_k 제외한 리스트 생성
# 테스트 > validation_k만 생성
class KCVDataset(Dataset):
    # 초기화: 데이터셋 전처리
    def __init__(self, K_CV, validation_k, save_path, is_train=True):
        if is_train:
            self.train_part = [i for i in range(K_CV) if i != validation_k]
        else:
            self.train_part = [validation_k]

        # 기본 저장 경로: save_path
        self.save_path = save_path

        # 반복문 > 파일 합 targets/embeddings 딕셔너리
        self.targets = {}
        self.embeddings = {}
        for i in self.train_part:
            # x, y 경로 설정
            path_x = os.path.join(self.save_path, f"embeddings_part_{i}.safetensors")
            path_y = os.path.join(self.save_path, f"target_part_{i}.pt")

            # 타겟 파일 로드
            y = torch.load(path_y, weights_only=False)
            self.targets.update(y)

            # 임베딩 파일
            with safe_open(path_x, framework="pt", device="cpu") as f:
                x = {k: f.get_tensor(k).squeeze() for k in f.keys()}
                self.embeddings.update(x)

        # 키 리스트 반환
        self.key_list = list(self.targets)

    # 몇개있는지 알려줘야함
    def __len__(self):
        return len(self.key_list)

    # 데이터셋 샘플 1개 가져오기
    def __getitem__(self, idx):
        # 리스트에서 idx번째 key 호출
        key = self.key_list[idx]

        # 키에 해당하는 임베딩, 타겟 불러오기
        embedding = self.embeddings[key].clone()
        target = torch.from_numpy(self.targets[key]).squeeze(0).float()

        return embedding, target


# 작동 테스트
# c = K_CV_MultipleFiles_Dataset(4, 0, SAVE_PATH)
# a, b = c[0]
# print(a)
# print(b)

In [ ]:
# 데이터셋 클래스 정의
class LMDBDataset_K_CV(Dataset):
    # 데이터셋 전처리
    def __init__(self, K_CV, test_k, save_path, is_train=True):
        if is_train:
            self.train_part = [i for i in range(K_CV) if i != test_k]
        else:
            self.train_part = [test_k]
        # 기본 저장 경로: save_path
        self.save_path = save_path
        # lmdb에서 key list 호출
        self.env = lmdb.open(
            self.save_path, readonly=True, lock=False, readahead=False, meminit=False
        )
        with self.env.begin() as txn:
            self.keys = pickle.loads(txn.get(b"__keys__"))

    # 몇개있는지 알려줘야함
    def __len__(self):
        return len(self.keys)

    # 데이터셋 샘플 1개 가져오기
    def __getitem__(self, idx):
        key = self.keys[idx]
        with self.env.begin() as txn:
            data = torch.load(io.BytesIO(txn.get(key.encode())))
        return data["embedding"], data["target"]


# c = LMDBDataset_K_CV(4, 0, SAVE_PATH)
# a, b = c[0]
# print(a)
# print(b)